In [13]:
import warnings
warnings.simplefilter(action='ignore')

import os
import datetime, pandas

from tqdm import tqdm
from dataclasses import dataclass, asdict

import polars as pl
import pandas as pd
import numpy as np
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple, Union
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import TimeSeriesSplit
import warnings
from abc import ABC, abstractmethod

import kaggle_evaluation.default_inference_server

from catboost import CatBoostRegressor, Pool
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.linear_model import ElasticNetCV
from sklearn.linear_model import LassoCV
from sklearn.linear_model import RidgeCV
from sklearn.linear_model import LassoLars
from sklearn.linear_model import Lasso
from sklearn.linear_model import BayesianRidge
from sklearn.linear_model import TweedieRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression

from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.cross_decomposition import PLSRegression
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestCentroid

from sklearn.naive_bayes import GaussianNB
from sklearn.naive_bayes import MultinomialNB
from sklearn.naive_bayes import BernoulliNB
from sklearn.naive_bayes import CategoricalNB

from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import ExtraTreesRegressor

from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import BaggingRegressor
from sklearn.ensemble import VotingRegressor
from sklearn.ensemble import StackingRegressor

from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.feature_selection import SelectKBest, f_classif, SelectFromModel
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor

In [14]:
train = pandas.read_csv('/kaggle/input/hull-tactical-market-prediction/train.csv').dropna()
test = pandas.read_csv('/kaggle/input/hull-tactical-market-prediction/test.csv').dropna()

In [15]:
train.columns

Index(['date_id', 'D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'E1',
       'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19',
       'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9', 'I1', 'I2', 'I3',
       'I4', 'I5', 'I6', 'I7', 'I8', 'I9', 'M1', 'M10', 'M11', 'M12', 'M13',
       'M14', 'M15', 'M16', 'M17', 'M18', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7',
       'M8', 'M9', 'P1', 'P10', 'P11', 'P12', 'P13', 'P2', 'P3', 'P4', 'P5',
       'P6', 'P7', 'P8', 'P9', 'S1', 'S10', 'S11', 'S12', 'S2', 'S3', 'S4',
       'S5', 'S6', 'S7', 'S8', 'S9', 'V1', 'V10', 'V11', 'V12', 'V13', 'V2',
       'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'forward_returns',
       'risk_free_rate', 'market_forward_excess_returns'],
      dtype='object')

In [16]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.impute import KNNImputer
import warnings

def preprocessing(data, typ, 
                 imputation_strategy='knn', 
                 scaling_method='robust',
                 handle_outliers=True,
                 outlier_threshold=3,
                 feature_engineering=True,
                 verbose=True):
    
    # Define main features with proper organization
    feature_groups = {
        'market_dynamics': ['M' + str(i) for i in range(1, 19)],
        'economic': ['E' + str(i) for i in range(1, 21)],
        'interest_rate': ['I' + str(i) for i in range(1, 10)],
        'price_valuation': ['P8', 'P9', 'P10', 'P12', 'P13'],
        'volatility': ['V' + str(i) for i in range(1, 14)],
        'sentiment': ['S' + str(i) for i in range(1, 13)],
        'dummy': ['D' + str(i) for i in range(1, 10)]
    }
    
    # Flatten all features
    main_features = []
    for group_features in feature_groups.values():
        main_features.extend(group_features)
    
    # Select relevant columns
    if typ == "train":
        target_col = "market_forward_excess_returns"
        if target_col in data.columns:
            selected_cols = main_features + [target_col]
        else:
            warnings.warn(f"Target column '{target_col}' not found in training data")
            selected_cols = main_features
    else:
        selected_cols = main_features
    
    # Filter for existing columns only
    available_cols = [col for col in selected_cols if col in data.columns]
    missing_cols = set(selected_cols) - set(available_cols)
    
    if missing_cols and verbose:
        print(f"Warning: Missing columns: {missing_cols}")
    
    data_processed = data[available_cols].copy()
    
    if verbose:
        print(f"Processing {len(data_processed)} rows with {len(available_cols)} features")
        print(f"Missing values before processing: {data_processed.isnull().sum().sum()}")
    
    # Separate features and target
    if typ == "train" and target_col in data_processed.columns:
        features = [col for col in data_processed.columns if col != target_col]
        X = data_processed[features]
        y = data_processed[target_col]
    else:
        features = list(data_processed.columns)
        X = data_processed[features]
        y = None
    
    # Store metadata
    metadata = {'feature_groups': feature_groups}
    
    # Handle missing values
    if imputation_strategy == 'zero':
        X = X.fillna(0)
    elif imputation_strategy == 'median':
        X = X.fillna(X.median())
    elif imputation_strategy == 'forward_fill':
        X = X.fillna(method='ffill').fillna(0)  # Forward fill then zero for remaining
    elif imputation_strategy == 'knn':
        # Use KNN imputation for more sophisticated missing value handling
        imputer = KNNImputer(n_neighbors=5)
        feature_names = X.columns
        X_imputed = imputer.fit_transform(X)
        X = pd.DataFrame(X_imputed, columns=feature_names, index=X.index)
        metadata['imputer'] = imputer
    
    # Handle outliers using IQR method
    if handle_outliers:
        numeric_cols = X.select_dtypes(include=[np.number]).columns
        outlier_info = {}
        
        for col in numeric_cols:
            Q1 = X[col].quantile(0.25)
            Q3 = X[col].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - outlier_threshold * IQR
            upper_bound = Q3 + outlier_threshold * IQR
            
            # Count outliers
            outliers = ((X[col] < lower_bound) | (X[col] > upper_bound)).sum()
            if outliers > 0:
                outlier_info[col] = outliers
                # Cap outliers instead of removing them
                X[col] = X[col].clip(lower=lower_bound, upper=upper_bound)
        
        if verbose and outlier_info:
            print(f"Capped outliers in {len(outlier_info)} columns: {dict(list(outlier_info.items())[:5])}")
        
        metadata['outlier_info'] = outlier_info
    
    # Feature Engineering
    if feature_engineering:
        # Create rolling statistics for time-sensitive features
        if 'date_id' in data.columns:
            # Sort by date for proper time series operations
            data_with_date = data.set_index('date_id').sort_index()
            
            # Rolling features for key metrics (example with market dynamics)
            market_cols = [col for col in X.columns if col.startswith('M')]
            for col in market_cols[:5]:  # Limit to avoid too many features
                if col in X.columns:
                    X[f'{col}_ma5'] = data_with_date[col].rolling(5).mean()
                    X[f'{col}_std5'] = data_with_date[col].rolling(5).std()
        
        # Create interaction features between different groups
        # Example: Volatility * Economic indicators
        vol_cols = [col for col in X.columns if col.startswith('V')][:3]
        econ_cols = [col for col in X.columns if col.startswith('E')][:3]
        
        for v_col in vol_cols:
            for e_col in econ_cols:
                if v_col in X.columns and e_col in X.columns:
                    X[f'{v_col}_{e_col}_interaction'] = X[v_col] * X[e_col]
        
        if verbose:
            print(f"Added feature engineering - new shape: {X.shape}")
    
    # Feature Scaling
    scaler = None
    if scaling_method == 'standard':
        scaler = StandardScaler()
    elif scaling_method == 'robust':
        scaler = RobustScaler()
    elif scaling_method == 'minmax':
        from sklearn.preprocessing import MinMaxScaler
        scaler = MinMaxScaler()
    
    if scaler is not None:
        # Only scale numeric columns
        numeric_cols = X.select_dtypes(include=[np.number]).columns
        X_scaled = X.copy()
        X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])
        X = X_scaled
        metadata['scaler'] = scaler
    
    # Combine features and target back
    if y is not None:
        result = pd.concat([X, y], axis=1)
    else:
        result = X
    
    # Final data quality check
    if verbose:
        print(f"Final shape: {result.shape}")
        print(f"Missing values after processing: {result.isnull().sum().sum()}")
        print(f"Data types: {result.dtypes.value_counts().to_dict()}")
    
    return result, metadata


processed_data, metadata = preprocessing(train, 'train')


Processing 2021 rows with 87 features
Missing values before processing: 0
Capped outliers in 35 columns: {'M1': 1, 'M3': 47, 'M4': 29, 'M6': 59, 'M9': 5}
Added feature engineering - new shape: (2021, 105)
Final shape: (2021, 106)
Missing values after processing: 40
Data types: {dtype('float64'): 106}


In [19]:

def preprocessing(data, typ):
    main_featurs = ['D1', 'D2', 'D3', 'D4', 'D5', 'D6', 'D7', 'D8', 'D9', 'E1',
                    'E10', 'E11', 'E12', 'E13', 'E14', 'E15', 'E16', 'E17', 'E18', 'E19',
                    'E2', 'E20', 'E3', 'E4', 'E5', 'E6', 'E7', 'E8', 'E9',
                    
                    'I1', 'I2', 'I3', 'I4', 'I5', 'I6', 'I7', 'I8', 'I9',
                    'M1', 'M10', 'M11', 'M12', 'M13','M14', 'M15', 'M16', 'M17', 'M18', 'M2', 
                    'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9',
                    
                    'S1', 'S10', 'S11', 'S12', 'S2', 'S3', 'S4',
                    'S5', 'S6', 'S7', 'S8', 'S9', 
                    
                    'V1', 'V10', 'V11', 'V12', 'V13', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9',
                    "P9", "P8", "P10", "P12", "P13"]
    
    if typ == "train":
        data = data[main_featurs + ["market_forward_excess_returns"]]
    else:
        data = data[main_featurs]
    
    for i in zip(data.columns, data.dtypes):
        data[i[0]].fillna(0, inplace=True)

    return data

train = preprocessing(train, "train")

train_split, val_split = train_test_split(
    train, test_size=0.01, random_state=42
)

X_train = train_split.drop(columns=["market_forward_excess_returns"])
X_test = val_split.drop(columns=["market_forward_excess_returns"])
y_train = train_split['market_forward_excess_returns']
y_test = val_split['market_forward_excess_returns']

In [20]:
X_train

,D1,D2,D3,D4,D5,D6,D7,D8,D9,E1,...,V5,V6,V7,V8,V9,P9,P8,P10,P12,P13
7206,0,0,0,0,0,0,0,0,0,1.041918,...,1.753795,0.363757,-1.018301,0.983796,-0.872875,0.033069,2.057248,2.023172,-0.376245,0.192791
7034,0,0,0,1,1,0,0,1,0,0.615708,...,-0.373969,0.073413,-1.161372,1.000000,-1.233493,0.000661,1.970050,1.914912,-0.151011,0.007275
7093,0,0,0,1,0,0,0,1,0,0.556396,...,1.581927,0.677910,0.239938,0.962632,0.024961,0.124339,1.898016,2.164918,-0.580531,0.384259
7588,0,0,0,1,0,0,0,0,0,2.446270,...,0.492018,0.140212,-0.199066,0.423280,-0.335226,0.011574,1.706585,2.116923,-0.447031,0.756944
7552,0,0,0,1,0,-1,0,0,0,2.141896,...,2.584326,0.256614,-0.607342,0.465939,-0.449239,0.071429,1.625190,1.983755,-0.838070,0.918651
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8099,0,0,0,1,0,-1,0,0,0,1.345860,...,-0.413963,0.141534,1.575796,0.708664,1.480310,0.001984,2.318440,2.141596,0.602239,0.218585
8263,0,0,0,1,0,0,0,0,0,1.382723,...,-1.450682,0.287037,1.297383,0.519511,1.262612,0.016534,1.186474,1.541084,-0.729057,0.116402
7829,0,0,0,1,1,0,0,1,0,1.347717,...,0.967203,0.217593,1.524194,0.760582,1.729834,0.082011,1.274399,1.610140,-0.781115,0.781415
8428,0,0,0,0,0,0,0,0,0,1.472010,...,1.665209,0.918651,-0.678125,0.133929,-0.665850,0.060185,2.295641,2.219362,-0.710188,0.669643


In [ ]:
improved_catboost_params = {'iterations': 3000,
                            'learning_rate': 0.01,
                            'depth': 6,
                            'l2_leaf_reg': 5.0,
                            'min_child_samples': 100,
                            'colsample_bylevel': 0.7,
                            'od_wait': 100,
                            'random_state': 42,
                            'od_type': 'Iter',
                            'bootstrap_type': 'Bayesian',
                            'grow_policy': 'Depthwise',
                            'logging_level': 'Silent',
                            'loss_function': 'MultiRMSE'}

R_Forest_parm = {'n_estimators': 100,
                 'min_samples_split': 5,
                 'max_depth': 15,
                 'min_samples_leaf': 3,
                 'max_features': 'sqrt',
                 'random_state': 42}
        
Extra_parm = {'n_estimators': 100,
              'min_samples_split': 5,
              'max_depth': 12,
              'min_samples_leaf': 3,
              'max_features': 'sqrt',
              'random_state': 42}
        
XGB_R_parm = {"n_estimators": 1500,
              "learning_rate": 0.05, 
              "max_depth": 6,
              "subsample": 0.8, 
              "colsample_bytree": 0.7,
              "reg_alpha": 1.0,
              "reg_lambda": 1.0,
              "random_state": 42}

LGBM_R_parm = {"n_estimators": 1500,
               "learning_rate": 0.05,
               "num_leaves": 50,
               "max_depth": 8,
               "reg_alpha": 1.0,
               "reg_lambda": 1.0,
               "random_state": 42,
               'verbosity': -1}

DecisionTree = {'criterion': 'poisson',
                'max_depth': 6}

GB_parm = {"learning_rate": 0.1,
           "min_samples_split": 500,
           "min_samples_leaf": 50,
           "max_depth": 8,
           "max_features": 'sqrt',
           "subsample": 0.8,
           "random_state": 10}

CatBoost = CatBoostRegressor(**improved_catboost_params)
XGBoost = XGBRegressor(**XGB_R_parm)
LGBM = LGBMRegressor(**LGBM_R_parm)
RandomForest = RandomForestRegressor(**R_Forest_parm)
ExtraTrees = ExtraTreesRegressor(**Extra_parm)
GBRegressor = GradientBoostingRegressor(**GB_parm)
        
DTRegressor = DecisionTreeRegressor(**DecisionTree)


estimators = [('CatBoost', CatBoost), ('XGBoost', XGBoost), ('LGBM', LGBM), ('RandomForest', RandomForest),
              ('ExtraTrees', ExtraTrees), ('GBRegressor', GBRegressor), 
              ('ElasticNet', ElasticNetCV()), ('Lasso', Lasso()), ('RidgeCV', RidgeCV()),
              ('LassoCV', LassoCV()), ('LassoLars', LassoLars())]

model = StackingRegressor(estimators, 
                          final_estimator = RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0]), 
                          cv=3)
model.fit(X_train, y_train)

In [ ]:
import numpy as np
import pandas as pd
import pandas.api.types

MIN_INVESTMENT = 0
MAX_INVESTMENT = 2


class ParticipantVisibleError(Exception):
    pass


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """
    Calculates a custom evaluation metric (volatility-adjusted Sharpe ratio).

    This metric penalizes strategies that take on significantly more volatility
    than the underlying market.

    Returns:
        float: The calculated adjusted Sharpe ratio.
    """

    if not pandas.api.types.is_numeric_dtype(submission['prediction']):
        raise ParticipantVisibleError('Predictions must be numeric')

    solution = solution
    solution['position'] = submission['prediction']

    if solution['position'].max() > MAX_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].max()} exceeds maximum of {MAX_INVESTMENT}')
    if solution['position'].min() < MIN_INVESTMENT:
        raise ParticipantVisibleError(f'Position of {solution["position"].min()} below minimum of {MIN_INVESTMENT}')

    solution['strategy_returns'] = solution['risk_free_rate'] * (1 - solution['position']) + solution['position'] * solution['forward_returns']

    # Calculate strategy's Sharpe ratio
    strategy_excess_returns = solution['strategy_returns'] - solution['risk_free_rate']
    strategy_excess_cumulative = (1 + strategy_excess_returns).prod()
    strategy_mean_excess_return = (strategy_excess_cumulative) ** (1 / len(solution)) - 1
    strategy_std = solution['strategy_returns'].std()

    trading_days_per_yr = 252
    if strategy_std == 0:
        raise ParticipantVisibleError('Division by zero, strategy std is zero')
    sharpe = strategy_mean_excess_return / strategy_std * np.sqrt(trading_days_per_yr)
    strategy_volatility = float(strategy_std * np.sqrt(trading_days_per_yr) * 100)

    # Calculate market return and volatility
    market_excess_returns = solution['forward_returns'] - solution['risk_free_rate']
    market_excess_cumulative = (1 + market_excess_returns).prod()
    market_mean_excess_return = (market_excess_cumulative) ** (1 / len(solution)) - 1
    market_std = solution['forward_returns'].std()

    market_volatility = float(market_std * np.sqrt(trading_days_per_yr) * 100)

    if market_volatility == 0:
        raise ParticipantVisibleError('Division by zero, market std is zero')

    # Calculate the volatility penalty
    excess_vol = max(0, strategy_volatility / market_volatility - 1.2) if market_volatility > 0 else 0
    vol_penalty = 1 + excess_vol

    # Calculate the return penalty
    return_gap = max(
        0,
        (market_mean_excess_return - strategy_mean_excess_return) * 100 * trading_days_per_yr,
    )
    return_penalty = 1 + (return_gap**2) / 100

    # Adjust the Sharpe ratio by the volatility and return penalty
    adjusted_sharpe = sharpe / (vol_penalty * return_penalty)
    return min(float(adjusted_sharpe), 1_000_000)

In [ ]:
def predict(test: pl.DataFrame) -> float:
    test = test.to_pandas()
    test = preprocessing(test, "test")
    raw_pred = model.predict(test)[0]
    return raw_pred

inference_server = kaggle_evaluation.default_inference_server.DefaultInferenceServer(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(('/kaggle/input/hull-tactical-market-prediction/',))